# VideoPrism Video-Text Encoder Demo

[![Paper](https://img.shields.io/badge/arXiv-2402.13217-red.svg)](https://arxiv.org/abs/2402.13217)
[![Blog](https://img.shields.io/badge/Google_Research-Blog-green.svg)](https://research.google/blog/videoprism-a-foundational-visual-encoder-for-video-understanding/)
[![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)

This notebook provides an example of video and text feature extraction with a pre-trained VideoPrism video-text model for zero-shot video classification/retrieval.

Please run this demo on Google Colab with (faster) or without TPU.

## Set up

In [2]:
# # @title Prepare environment

# import os, sys, subprocess

# # Fetch VideoPrism repository if Python does not know about it and install
# # dependencies needed for this notebook.
# if not os.path.exists("videoprism_repo"):
#   !git clone --quiet --branch=main --depth=1 \
#      https://github.com/google-deepmind/videoprism.git videoprism_repo
#   os.chdir('./videoprism_repo')
#   # If tensorflow is installed, remove the last line of requirements.txt to avoid version conflict
#   result = subprocess.run(['pip', 'show', 'tensorflow'], capture_output=True)
#   if result.returncode == 0:
#     !sed -i '$d' requirements.txt
#   !pip install .
#   os.chdir('..')

# # Append VideoPrism code to Python import path.
# if "videoprism_repo" not in sys.path:
#   sys.path.append("videoprism_repo")

# # Install missing dependencies.
# !pip install mediapy
# !pip install jax

# import jax
# from jax.extend import backend
# import tensorflow as tf

# # Do not let TF use the GPU or TPUs.
# tf.config.set_visible_devices([], "GPU")
# tf.config.set_visible_devices([], "TPU")

# print(f"JAX version:  {jax.__version__}")
# print(f"JAX platform: {backend.get_backend().platform}")
# print(f"JAX devices:  {jax.device_count()}")

In [3]:
import os

In [4]:
import mediapy
import numpy as np
from PIL import Image


def read_and_preprocess_video(
    filename: str, target_num_frames: int, target_frame_size: tuple[int, int]
):
  """Reads and preprocesses a video, adjusting aspect ratio if necessary."""

  frames = mediapy.read_video(filename)

  # Sample to target number of frames.
  frame_indices = np.linspace(
      0, len(frames), num=target_num_frames, endpoint=False, dtype=np.int32
  )
  frames = np.array([frames[i] for i in frame_indices])

  # Resize to target size, adjusting aspect ratio by cropping if needed.
  original_height, original_width = frames.shape[-3:-1]
  target_height, target_width = target_frame_size

  # Calculate aspect ratios
  original_aspect_ratio = original_width / original_height
  target_aspect_ratio = target_width / target_height

  # Check if aspect ratios are significantly different
  if abs(original_aspect_ratio - target_aspect_ratio) > 1e-6:
    if original_aspect_ratio > target_aspect_ratio:
      # Video is wider than target aspect ratio, crop width
      new_width = int(original_height * target_aspect_ratio)
      offset = (original_width - new_width) // 2
      frames = frames[:, :, offset:offset + new_width, :]
    else:
      # Video is taller than target aspect ratio, crop height
      new_height = int(original_width / target_aspect_ratio)
      offset = (original_height - new_height) // 2
      frames = frames[:, offset:offset + new_height, :, :]

  # Now that aspect ratios match, resize to target_frame_size
  frames = mediapy.resize_video(frames, shape=target_frame_size)

  # Normalize pixel values to [0.0, 1.0].
  frames = mediapy.to_float01(frames)

  return frames


def compute_similarity_matrix(
    video_embeddings,
    text_embeddings,
    temperature: float,
    apply_softmax: str | None = None,
) -> np.ndarray:
  """Computes cosine similarity matrix."""
  assert apply_softmax in [None, 'over_texts', 'over_videos']
  emb_dim = video_embeddings[0].shape[-1]
  assert emb_dim == text_embeddings[0].shape[-1]

  video_embeddings = np.array(video_embeddings).reshape(-1, emb_dim)
  text_embeddings = np.array(text_embeddings).reshape(-1, emb_dim)
  similarity_matrix = np.dot(video_embeddings, text_embeddings.T)

  if temperature is not None:
    similarity_matrix /= temperature

  if apply_softmax == 'over_videos':
    similarity_matrix = np.exp(similarity_matrix)
    similarity_matrix = similarity_matrix / np.sum(
        similarity_matrix, axis=0, keepdims=True
    )
  elif apply_softmax == 'over_texts':
    similarity_matrix = np.exp(similarity_matrix)
    similarity_matrix = similarity_matrix / np.sum(
        similarity_matrix, axis=1, keepdims=True
    )

  return similarity_matrix

In [5]:
# @title Load model

import jax
import jax.numpy as jnp
from videoprism import models as vp

MODEL_NAME = 'videoprism_lvt_public_v1_base'  # @param ['videoprism_lvt_public_v1_base', 'videoprism_lvt_public_v1_large'] {allow-input: false}
USE_BFLOAT16 = True  # @param { type: "boolean" }
NUM_FRAMES = 16
FRAME_SIZE = 288

fprop_dtype = jnp.bfloat16 if USE_BFLOAT16 else None
flax_model = vp.get_model(MODEL_NAME, fprop_dtype=fprop_dtype)
loaded_state = vp.load_pretrained_weights(MODEL_NAME)
text_tokenizer = vp.load_text_tokenizer('c4_en')


@jax.jit
def forward_fn(inputs, text_token_ids, text_paddings, train=False):
  return flax_model.apply(
      loaded_state,
      inputs,
      text_token_ids,
      text_paddings,
      train=train,
  )

# Example: Zero-shot Video Classification/Retrieval

In this example, we extract the embedding of an input video, and the embeddings of five senetence. We measure the cosine similarites between the videos and sentences.

In [6]:
EMOTION_PROMPTS = {
    "joy": (
        "A social-media image or video conveying strong joy, happiness, and positive energy. "
        "The emotion may be expressed through people smiling, laughing, celebrating, playing, "
        "hugging, spending time together, enjoying food or activities, or through positive scenes "
        "such as beautiful nature, colorful surroundings, pets, achievements, festivals, weddings, "
        "success, friendship, family moments, or uplifting events. Use bright and warm visual tones, "
        "lively activity, open body language, energetic movement, pleasant surroundings, and an overall "
        "sense of celebration, pleasure, connection, optimism, and emotional warmth. The scene does "
        "not require a person; objects, environments, events, and visual context can communicate joy."
    ),

    "sadness": (
        "A social-media image or video conveying sadness, sorrow, grief, loneliness, disappointment, "
        "or emotional pain. The emotion may be expressed through crying or distressed people, but can "
        "also appear through empty or abandoned places, separation, loss, funerals, memorials, damaged "
        "homes, lonely individuals, rainy or gloomy environments, neglected spaces, fading memories, "
        "or situations involving failure or hardship. Use subdued colors, low-energy compositions, "
        "downcast or isolated subjects, empty spaces, stillness, and melancholic atmosphere where "
        "appropriate. The overall context should communicate emotional heaviness, loss, loneliness, "
        "helplessness, or sorrow even when no person is visible."
    ),

    "anger": (
        "A social-media image or video conveying anger, rage, outrage, hostility, frustration, or "
        "strong opposition. The emotion may be expressed through visibly angry people, arguments, "
        "confrontations, protests, shouting, aggressive gestures, or tense crowds, but can also be "
        "communicated through scenes of injustice, destruction, violence, corruption, unfair treatment, "
        "political or social conflict, property damage, or other situations that provoke outrage. "
        "Look for tense body language, confrontational interactions, clenched fists, aggressive gestures, "
        "chaotic movement, harsh visual composition, warning signs, damaged objects, or visibly hostile "
        "situations. The emotion should feel intense, confrontational, and negative rather than merely "
        "serious or concerned."
    ),

    "fear": (
        "A social-media image or video conveying fear, anxiety, panic, danger, threat, or insecurity. "
        "The emotion may be expressed through frightened or fleeing people, but can also be communicated "
        "by dangerous environments, natural disasters, accidents, fires, storms, floods, war, threatening "
        "animals, unsafe situations, darkness, destruction, or scenes suggesting imminent danger. "
        "Use visual cues such as people running or hiding, defensive body language, chaotic situations, "
        "dark or threatening surroundings, emergency conditions, destruction, uncertainty, or isolation. "
        "The overall context should suggest vulnerability, danger, alarm, or a desire to escape or seek safety, "
        "even when no person is present."
    ),

    "surprise": (
        "A social-media image or video conveying surprise, astonishment, shock, amazement, or an unexpected "
        "discovery. The emotion may be expressed through people with wide eyes, open mouths, startled reactions, "
        "or sudden gestures, but can also come from unexpected events, unusual objects, dramatic transformations, "
        "rare natural phenomena, unexpected outcomes, sudden accidents, extraordinary achievements, shocking news, "
        "or visually unusual situations. Emphasize a clear contrast between what would normally be expected and "
        "what is actually happening. The scene should communicate an immediate sense of something unexpected, "
        "unbelievable, or astonishing, whether or not people are visible."
    ),

    "disgust": (
        "A social-media image or video conveying disgust, revulsion, nausea, repulsion, or strong aversion. "
        "The emotion may be expressed through people grimacing, covering their noses, turning away, or reacting "
        "negatively, but can also be communicated directly through unpleasant or contaminated scenes such as "
        "rotting food, garbage, pollution, sewage, unhygienic conditions, spoiled substances, infestation, "
        "severe neglect, disturbing contamination, or visibly revolting environments. Use visual cues such as "
        "dirty or decaying surroundings, unpleasant textures, contamination, foul-looking substances, people "
        "recoiling, or objects being avoided. The overall scene should communicate a strong desire to reject, "
        "avoid, or distance oneself from something unpleasant."
    ),

    "trust": (
        "A social-media image or video conveying trust, safety, reassurance, confidence, acceptance, peace, "
        "and emotional security. The emotion may be expressed through people helping, supporting, cooperating, "
        "comforting, caring for one another, forming strong relationships, or interacting peacefully, but can "
        "also be communicated through safe and welcoming environments, reliable services, community cooperation, "
        "protective actions, responsible organizations, successful teamwork, or positive relationships between "
        "people and animals. Use relaxed body language, open interactions, supportive gestures, calm environments, "
        "stable compositions, welcoming surroundings, and visual signs of reliability and security. The overall "
        "context should communicate safety, dependability, care, reassurance, or confidence rather than simple "
        "happiness."
    ),

    "anticipation": (
        "A social-media image or video conveying anticipation, eagerness, expectation, hope, curiosity, or "
        "excitement about something that is about to happen. The emotion may be expressed through people waiting, "
        "preparing, looking toward an expected event, gathering before a celebration, watching a countdown, "
        "opening a package, preparing for travel, awaiting results, or getting ready for an important occasion. "
        "It can also be communicated without people through visual clues such as unopened packages, event setups, "
        "stages prepared for an audience, countdowns, starting lines, preparations, announcements, unfinished "
        "reveals, or scenes suggesting an upcoming event. The key characteristic is that something meaningful "
        "or exciting is expected to happen in the near future, creating a sense of curiosity, eagerness, tension, "
        "or hopeful excitement."
    ),
}

In [7]:
emotion_desc = [f"{emotion}: {desc}"for emotion,desc in EMOTION_PROMPTS.items()]

In [8]:
# @title Specify input text queries
# TEXT_QUERY_CSV = 'playing drums,sitting,playing flute,playing at playground,concert'  # @param {type: "string"}

PROMPT_TEMPLATE = 'a video of {}.'

# text_queries = TEXT_QUERY_CSV.split(',')
text_queries = [PROMPT_TEMPLATE.format(t) for t in emotion_desc]
text_ids, text_paddings = vp.tokenize_texts(text_tokenizer, text_queries)
if USE_BFLOAT16:
  text_paddings = text_paddings.astype(jnp.bfloat16)

print('Input text queries:')
for i, text in enumerate(text_queries):
  print(f'({i + 1}) {text}')

Input text queries:
(1) a video of joy: A social-media image or video conveying strong joy, happiness, and positive energy. The emotion may be expressed through people smiling, laughing, celebrating, playing, hugging, spending time together, enjoying food or activities, or through positive scenes such as beautiful nature, colorful surroundings, pets, achievements, festivals, weddings, success, friendship, family moments, or uplifting events. Use bright and warm visual tones, lively activity, open body language, energetic movement, pleasant surroundings, and an overall sense of celebration, pleasure, connection, optimism, and emotional warmth. The scene does not require a person; objects, environments, events, and visual context can communicate joy..
(2) a video of sadness: A social-media image or video conveying sadness, sorrow, grief, loneliness, disappointment, or emotional pain. The emotion may be expressed through crying or distressed people, but can also appear through empty or ab

In [9]:
# @title Specify input video
VIDEO_FILE_PATH = 'videoprism_repo/videoprism/assets/joy_3_1.mp4'  # @param {type: "string"}

frames = read_and_preprocess_video(
    VIDEO_FILE_PATH,
    target_num_frames=NUM_FRAMES,
    target_frame_size=[FRAME_SIZE, FRAME_SIZE],
)
frames = jnp.asarray(frames[None, ...])  # Add batch dimension.
if USE_BFLOAT16:
  frames = frames.astype(jnp.bfloat16)

In [10]:
# @title Compute video-to-text retrieval results
video_embeddings, text_embeddings, _ = forward_fn(
    frames, text_ids, text_paddings)

TEMPERATURE = 0.01  # @param {type: "number"}
similarity_matrix = compute_similarity_matrix(
    video_embeddings,
    text_embeddings,
    temperature=TEMPERATURE,
    apply_softmax='over_texts',
)

In [11]:
v2t_similarity_vector = similarity_matrix[0]
top_indices = np.argsort(v2t_similarity_vector)[::-1]

print(f'Query video: {os.path.basename(VIDEO_FILE_PATH)}')
mediapy.show_video(frames[0].astype(jnp.float32), fps=6.0)

for k, j in enumerate(top_indices):
  print(
      'Top-%d retrieved text: %s [Similarity = %0.4f]'
      % (k + 1, text_queries[j].split(":")[0], v2t_similarity_vector[j])
  )
print(f'\nThis is {text_queries[top_indices[0]]}')

Query video: joy_3_1.mp4


This browser does not support the video tag.


Top-1 retrieved text: a video of joy [Similarity = 0.9922]
Top-2 retrieved text: a video of trust [Similarity = 0.0086]
Top-3 retrieved text: a video of surprise [Similarity = 0.0016]
Top-4 retrieved text: a video of anticipation [Similarity = 0.0009]
Top-5 retrieved text: a video of sadness [Similarity = 0.0005]
Top-6 retrieved text: a video of fear [Similarity = 0.0003]
Top-7 retrieved text: a video of disgust [Similarity = 0.0002]
Top-8 retrieved text: a video of anger [Similarity = 0.0001]

This is a video of joy: A social-media image or video conveying strong joy, happiness, and positive energy. The emotion may be expressed through people smiling, laughing, celebrating, playing, hugging, spending time together, enjoying food or activities, or through positive scenes such as beautiful nature, colorful surroundings, pets, achievements, festivals, weddings, success, friendship, family moments, or uplifting events. Use bright and warm visual tones, lively activity, open body language,